# Notebook

This notebook test finished code.

In [1]:
#%pip install -r requirements.txt

### **1. Load Article & Chunking**

In [2]:
from langchain_docling import DoclingLoader
from docling.chunking import HybridChunker

FILE_PATH = r"my_papers/2024 - 1.pdf"

loader = DoclingLoader(file_path=FILE_PATH, chunker=HybridChunker())

docs = loader.load()

[INFO] 2026-02-16 15:52:25,660 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-16 15:52:25,666 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-16 15:52:25,667 [RapidOCR] main.py:53: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-02-16 15:52:25,784 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-02-16 15:52:25,787 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-16 15:52:25,788 [RapidOCR] main.py:53: Using C:\Users\sahil\AppData\Roaming\Python\Python313\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-02-16 15:52:25,847 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 20

### **2. Embeddings and Vector Storage**

Option 1: In-memory vector Store and OpenAI embeddings

In [4]:
# If Export Type DOC_CHUNKS (Default)
doc_splits = docs

In [5]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits, embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [6]:
query = "Key question addressed in this study?"

In [7]:
retriever.invoke(query)

[Document(id='d3e47464-ab55-47d3-9fd4-912cbd25451f', metadata={'source': 'my_papers/2024 - 1.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/262', 'parent': {'$ref': '#/body'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': [{'page_no': 8, 'bbox': {'l': 37.587, 't': 706.8099887695312, 'r': 291.032, 'b': 553.0619887695312, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 1036]}]}], 'headings': ['5.2.1. Questionnaire collection'], 'origin': {'mimetype': 'application/pdf', 'binary_hash': 9586957159616026950, 'filename': '2024 - 1.pdf'}}}, page_content='5.2.1. Questionnaire collection\nThis  study  conducted  a  small-scale  field  pretest  in  Quanzhou  to evaluate the initial questionnaire \' s quality, and 51 valid questionnaires were collected by convenience sampling. According to the respondents \' feedback, the item readability was adjusted and improved, and the final formal questionn

Option 2: Chromadb

In [13]:
import chromadb

In [14]:
client = chromadb.Client()

In [15]:
collection = client.get_or_create_collection(
    name="my_collection"
)

In [16]:
# Add documents with embeddings
collection.add(
    ids=["id1", "id2"],
    documents=["This is a document", "Another doc"],
    embeddings=[[1.2, 2.3, ...], [3.4, 4.5, ...]]
)

TypeError: float() argument must be a string or a real number, not 'ellipsis' in add.

Will use custom-designed retrieval formatter.

In [ ]:
from modules.tools.retrieval_tools import make_retriever_tool

retriever_tool = make_retriever_tool(retriever)
result = retriever_tool.invoke(query)
print(result)

### LLMs



In [9]:
import os
from langchain.chat_models import init_chat_model

In [10]:
# Chat GPT mini
model = init_chat_model("gpt-4o-mini")

In [11]:
response = model.invoke("What is the capital of India?")

In [12]:
# response # See whats inside the response object

response.content # Simple Response

'The capital of India is New Delhi.'

## **LangGraph**

In [69]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import json

class State(TypedDict):
    query: str
    retrieved_docs: str
    response: str

def retrieve_node(state: State):
    """Call retrieval tool"""
    docs = retriever_tool.invoke(state["query"])
    return {"retrieved_docs": docs}


def llm_filter_node(state: State):
    """LLM filters and structures results as prose with citations"""
    llm = model
    
    # Parse JSON from retriever_tool
    try:
        docs_list = json.loads(state["retrieved_docs"])
        docs_str = "\n".join([
            f"[{doc['id']}] {doc['content']}\nProvenance: {doc['provenance']}"
            for doc in docs_list
        ])
    except:
        docs_str = state["retrieved_docs"]
    
    prompt = f"""Original Query: {state['query']}

Retrieved Documents:
{docs_str}

Based on the retrieved documents, write a well-structured paragraph that directly answers the query. 

Requirements:
- Write in clear, academic prose (not JSON)
- Use inline citations like [1], [2], [3] that match the document IDs above
- Only include information directly relevant to: {state['query']}
- Cite after each fact using the correct ID from retrieved documents
- End with Sources section: [ID] filename | page_no | heading

Ignore any chunks not directly relevant."""
    
    response = llm.invoke(prompt)
    return {"response": response.content}

# Build graph
graph = StateGraph(State)
graph.add_node("retrieve", retrieve_node)
graph.add_node("filter", llm_filter_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "filter")
graph.add_edge("filter", END)

app = graph.compile()


In [70]:
from IPython.display import Image, display

# Visualize the graph
#img = app.get_graph().draw_mermaid_png()
#display(Image(img))

In [71]:
result = app.invoke({
    "query": "Key question addressed in this study?",
    "retrieved_docs": "",
    "response": ""
})


In [26]:
print(result.keys())

#print(result["response"])

dict_keys(['query', 'retrieved_docs', 'response'])


In [72]:
from IPython.display import display, Markdown
display(Markdown(result["response"]))

The key question addressed in this study pertains to understanding the dynamics of cultural resilience, particularly in the context of cultural heritage sites during disturbances, such as the COVID-19 pandemic. The study aimed to explore how cultural elements familiar to respondents are preserved or developed in response to such shocks, as well as the specific processes and practices involved in addressing these disturbances [3]. To investigate this, a two-round questionnaire was designed, focusing on cultural resilience and related predictive variables, ensuring robust validity through both exploratory and confirmatory analyses [2][4]. The ultimate goal was to gather insights from various stakeholders, including tourists and local residents, regarding their perspectives on cultural resilience and the impact of modern challenges on cultural heritage [3] 

Sources: [1] 2024 - 1.pdf | page_no 8 | heading 5.2.1. Questionnaire collection  
[2] 2024 - 1.pdf | page_no 8 | heading 5.3.1. Questionnaire distribution and analysis  
[3] 2024 - 1.pdf | page_no 6 | heading 4.2.1. Open coding  
[4] 2024 - 1.pdf | page_no 4 | heading 3.2. Methods and design  

**[Optional] Testing a Iterative, Query re-writing  workflow**

In [73]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, List

class State(TypedDict):
    original_query: str
    query: str
    sub_tasks: List[str]
    retrieved_docs: str
    response: str
    iterations: int

def rewrite_query_node(state: State):
    """Rewrite query for better retrieval"""
    llm = model
    prompt = f"""Improve this query for better document retrieval:
Original: {state['original_query']}

Provide a clearer, more specific version that will retrieve relevant information."""
    
    rewritten = llm.invoke(prompt).content
    return {"query": rewritten}

def decompose_node(state: State):
    """Break complex query into sub-tasks"""
    llm = model
    prompt = f"""Break this query into 2-3 specific sub-tasks:
{state['query']}

Return as a list. Example:
1. What is X?
2. How does Y relate to Z?"""
    
    response = llm.invoke(prompt).content
    tasks = [task.strip() for task in response.split('\n') if task.strip()]
    return {"sub_tasks": tasks}

def retrieve_node(state: State):
    """Retrieve for current query or sub-task"""
    docs = retriever_tool.invoke(state["query"])
    return {"retrieved_docs": docs}

def process_sub_tasks_node(state: State):
    """Process each sub-task sequentially"""
    all_docs = []
    for task in state["sub_tasks"]:
        docs = retriever_tool.invoke(task)
        all_docs.append(docs)
    return {"retrieved_docs": "\n\n".join(all_docs)}

def filter_node(state: State):
    """LLM filters and structures"""
    llm = model
    prompt = f"""Query: {state['query']}
Retrieved: {state['retrieved_docs']}

Write structured prose with citations [1], [2], etc."""
    
    response = llm.invoke(prompt).content
    return {"response": response, "iterations": state.get("iterations", 0) + 1}

def should_iterate(state: State):
    """Decide if results are sufficient"""
    # You can add logic here - for now, single pass
    return "done"

# Build graph with routing
graph = StateGraph(State)
graph.add_node("rewrite", rewrite_query_node)
graph.add_node("decompose", decompose_node)
graph.add_node("retrieve", retrieve_node)
graph.add_node("process_subtasks", process_sub_tasks_node)
graph.add_node("filter", filter_node)

graph.add_edge(START, "rewrite")
graph.add_edge("rewrite", "decompose")
graph.add_conditional_edges(
    "decompose",
    lambda s: "process_subtasks" if len(s.get("sub_tasks", [])) > 1 else "retrieve"
)
graph.add_edge("retrieve", "filter")
graph.add_edge("process_subtasks", "filter")
graph.add_edge("filter", END)

app = graph.compile()


In [77]:
Query = "What was the nature of research question in this study? \
\
Exploratory, Descriptive, Predictive, Inferential, or Causal. \
You are rigorous qualitative researcher. \
Analyse the study and classify the research question into one of these categories. \
Provide a concise justification for your classification based on the content of the paper. \
Focus on the main research question and its underlying intent."


In [78]:

result = app.invoke({
    "original_query": Query,
    "query": "",
    "sub_tasks": [],
    "retrieved_docs": "",
    "response": "",
    "iterations": 0
})
#print(result["response"])

In [79]:
from IPython.display import display, Markdown
display(Markdown(result["response"]))

In evaluating the classification of the research question presented in this study, it becomes evident that the question can be categorized as **Exploratory**. The intent of the research appears to focus on uncovering new insights and gaining a deeper understanding of the practices and elements of cultural resilience, particularly in the context of heritage sites affected by disturbances such as the COVID-19 pandemic.

One of the main elements supporting this classification is the study's approach to data collection, which involves qualitative data gathered through field and online interviews. The interviews target various stakeholders in the tourism and heritage sector, inviting them to discuss their experiences and perceptions regarding cultural resilience. This technique aligns with exploratory research, which typically seeks to explore complex phenomena when there is limited pre-existing knowledge or understanding [1]. The questionnaire used in the study underscores this intent, as it encourages respondents to share their insights regarding the impacts of cultural disturbances and how these challenges are addressed, further reinforcing the exploratory nature of the inquiry [2].

The responses gathered through open coding and qualitative analysis suggest that the primary aim is to probe into the intricacies of cultural resilience and its manifestations in a specific context, namely, the ancient city of Quanzhou [3]. By enabling respondents to articulate their views and experiences, the researchers aim to compile a rich dataset that helps in formulating a comprehensive understanding of cultural heritage and resilience. Additionally, the study's iterative process of scale development and validation—from initial qualitative insights to subsequent quantitative analyses—suggests a commitment to refining and enriching the foundational constructs of cultural resilience rather than testing predetermined hypotheses, a hallmark of exploratory research [4].

Furthermore, the characterization of the study's goal—to inform future research directions—also supports this classification. By addressing gaps in the current understanding of cultural resilience and proposing new dimensions for consideration, the research fulfills a classic exploratory function within academic inquiry [5]. 

In conclusion, the exploratory classification of this research question is firmly supported by its methodological design, an emphasis on qualitative insights, and an underlying goal of uncovering new understandings rather than confirming existing theories or relationships. The research stands as a vital contribution to the ongoing discourse on cultural resilience, particularly in heritage settings.

Citations:
[1] Study methodology and data collection process.
[2] Questionnaire design and respondent engagement.
[3] Analysis of open coding techniques.
[4] Scale development and validation processes.
[5] Theoretical contributions and future research direction.

**Optional 2: Understanding how many iterations happened under the hood**

Rendering what happening

In [54]:
import json
from IPython.display import display, Markdown

def print_workflow_events(app, initial_state):
    """Print workflow events in a readable format"""
    print("=" * 80)
    print("WORKFLOW EXECUTION")
    print("=" * 80)
    
    for event in app.stream(initial_state):
        for node_name, node_output in event.items():
            print(f"\n📍 NODE: {node_name.upper()}")
            print("-" * 80)
            
            if node_name == "rewrite":
                print(f"✏️  Rewritten Query: {node_output['query'][:150]}...")
            
            elif node_name == "decompose":
                print(f"🔀 Sub-tasks ({len(node_output['sub_tasks'])} tasks):")
                for i, task in enumerate(node_output['sub_tasks'], 1):
                    print(f"   {i}. {task[:100]}...")
            
            elif node_name == "process_subtasks":
                docs_all = node_output['retrieved_docs']
                # Count JSON arrays in the string
                doc_count = docs_all.count('"id"')
                print(f"📄 Retrieved Documents: {doc_count} total chunks")
            
            elif node_name == "filter":
                response = node_output['response']
                print(f"✅ Response Generated:")
                display(Markdown(response))
            
            print()

# Use it:
initial_state = {
    "original_query": "What are the main findings of this study?",
    "query": "",
    "sub_tasks": [],
    "retrieved_docs": "",
    "response": "",
    "iterations": 0,
    "execution_history": []
}

print_workflow_events(app, initial_state)

WORKFLOW EXECUTION

📍 NODE: REWRITE
--------------------------------------------------------------------------------
✏️  Rewritten Query: Revised Query: "What are the key results and conclusions presented in the study regarding [specific topic or focus of the study]?" 

This version spec...


📍 NODE: DECOMPOSE
--------------------------------------------------------------------------------
🔀 Sub-tasks (3 tasks):
   1. 1. What are the key results outlined in the study regarding [specific topic or focus of the study]?...
   2. 2. What conclusions do the authors draw based on the findings related to [specific topic or focus of...
   3. 3. What implications or recommendations do the authors suggest for future research or practice based...


📍 NODE: PROCESS_SUBTASKS
--------------------------------------------------------------------------------
📄 Retrieved Documents: 12 total chunks


📍 NODE: FILTER
--------------------------------------------------------------------------------
✅ Response

The study presents several key findings and conclusions that contribute to the understanding of applying Large Language Models (LLM) in the tourism sector, focusing on the analysis of tourism reviews and resource management.

### Research Framework and Methodology
The research framework outlines the integration of LLMs, particularly ChatGPT, in analyzing tourism reviews, filling a notable gap in existing literature that predominantly emphasized qualitative analysis within the tourism domain. This updated methodology potentially enhances the understanding of consumer preferences and behavior in tourism contexts, especially in remote areas where traditional data analysis resources are scarce [1].

### Theoretical Implications
Two primary theoretical contributions are highlighted in the study. Firstly, it posits that the emergence of LLMs in natural language processing offers a revolutionary approach to analyze and interpret tourism-related texts, thereby addressing methodological inadequacies in current research [2]. Secondly, the research validates the applicability of LLM frameworks in resource-constrained settings, indicating that critical insights can be derived from reviews without requiring advanced AI expertise or significant computational resources [2].

### Practical Implications
The implications for managers in the tourism industry are significant. The study suggests that managers can utilize LLM tools to analyze large datasets of unlabeled reviews from various platforms, helping them identify critical themes and satisfaction factors. This analysis not only enables ongoing assessments of customer satisfaction across different venues such as hotels and restaurants but also facilitates comparative evaluations among similar destinations [2]. Furthermore, managers are encouraged to implement iterative improvements based on feedback and reassess the effectiveness of their strategies over time to achieve sustained enhancements in service quality [2].

### Addressing Limitations and Future Research Directions
The study also acknowledges certain limitations, including a constrained sample size, which suggests that future research should incorporate broader datasets for more comprehensive analyses. Additionally, the authors point out the initial nature of their framework so far as integrating sustainability measures with LLMs, highlighting that further methodological advancements are essential [3]. Furthermore, ChatGPT’s analytical speed and availability issues in some regions are noted as factors requiring attention in future work to facilitate better application and accessibility of AI tools in tourism research [3].

### Conclusion
In conclusion, the research effectively demonstrates the utility of LLMs in the tourism sector, outlining both theoretical advancements and practical applications for enhancing consumer sentiment analysis and resource management, while simultaneously addressing the need for addressing current limitations and exploring future research avenues [1][2][3].